In [8]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [9]:
load_dotenv()

True

In [10]:
model = ChatGoogleGenerativeAI(model='gemini-3.8-flash')

In [11]:

# create a state

class LLMState(TypedDict):

    question: str
    answer: str

In [12]:
def llm_qa(state: LLMState) -> LLMState:

    # extract the question from state
    question = state['question']

    # form a prompt
    prompt = f'Answer the following question {question}'

    # ask that question to the LLM
    answer = model.invoke(prompt).content

    # update the answer in the state
    state['answer'] = answer

    return state

In [13]:
# create our graph

graph = StateGraph(LLMState)

# add nodes
graph.add_node('llm_qa', llm_qa)

# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile
workflow = graph.compile()

In [14]:
# execute

intial_state = {'question': 'How far is moon from the earth?'}

final_state = workflow.invoke(intial_state)

print(final_state['answer'])

[{'type': 'text', 'text': 'On average, the Moon is about **238,855 miles (384,400 kilometers)** away from Earth.\n\nBecause the Moon travels in an elliptical (oval-shaped) orbit rather than a perfect circle, this distance changes over the course of a month:\n\n* **Closest point (Perigee):** about **225,623 miles (363,300 km)**\n* **Farthest point (Apogee):** about **251,966 miles (405,500 km)**\n\nTo put that distance into perspective, you could fit about 30 Earth-sized planets side-by-side in the space between the Earth and the Moon.', 'extras': {'signature': 'EvwFCvkFAWkUfRMk3ZS3KVYsYgiBu1TI8EFLBHEKidCB636kbfSBbJ7ICnPYpySF4Ag4MSrKIvUILjqGMRfbmiLdvLkCzmEexYJuNsSeYAS13qBDOLUtkP3m4inPiNk5GjdGgAXsu8ff5Q9F3tGu2P7LJpDo1ntVhbC7dFypmJsr2XlnQujvRa7pgnr1wWnjm/vDLAKaOWEwM1TdpGLsIJ451r/RKLJLzjuuAqnTCOEgc+IXAKuLHgFCc6zXxwiwQHZA/Ou4w7ehmXH9dmVb7xbhodjoCfLU2Q8NHYiZNTvr10HpDlBCEVIAeIxZgTSQBOZuiNYLTWgj/uWYpriamDYEDh+olx6nvXzDIvrCQ3NWcPs295EkBu2zkVedByyWlWITTdbHIATogUOg97MiRuboXR/yvifOS3qZ7Fn7V5S3AwOL